In [0]:
%run /Users/malkeet21.uk@gmail.com/Databricks-Data-Engineering-Assesment/common_utils/utils

#Task 4: Data analysis

In [0]:
%python
from pyspark.sql.functions import col

# Step 1: read the three silver tables and convert 90_and_over from text ("350,700") to integer (350700)
df_persons = spark.table(SILVER_PERSONS_TABLE).withColumn("90_and_over", to_int("90_and_over"))
df_males = spark.table(SILVER_MALES_TABLE).withColumn("90_and_over", to_int("90_and_over"))
df_females = spark.table(SILVER_FEMALES_TABLE).withColumn("90_and_over", to_int("90_and_over"))

"""clean Persons,male and female table from silver and replaces "350,700" with 350700"""

# Step 2: join Females and Males to Persons on year
# (rename each count first, because all three DataFrames have a column called 90_and_over)
persons = df_persons.select("year", col("90_and_over").alias("90_and_over_num_persons"))
females = df_females.select("year", col("90_and_over").alias("90_and_over_num_females"))
males = df_males.select("year", col("90_and_over").alias("90_and_over_num_males"))

df_joined = persons.join(females, on="year", how="inner").join(males, on="year", how="inner")
check_row_count(df_joined, YEARS_PER_POPULATION, "Joined")   # no year lost in the join

# Step 3: percentage split of 90_and_over between females and males, to 2 decimal places
df_split = (
    df_joined
    .withColumn("90_and_over_pcnt_females", percentage("90_and_over_num_females", "90_and_over_num_persons"))
    .withColumn("90_and_over_pcnt_males", percentage("90_and_over_num_males", "90_and_over_num_persons"))
)

# Step 4: display in the required column order, percentages shown as e.g. 76.87%
df_result = df_split.select(
    "year",
    "90_and_over_num_persons",
    as_percent_text("90_and_over_pcnt_females"),
    "90_and_over_num_females",
    as_percent_text("90_and_over_pcnt_males"),
    "90_and_over_num_males",
).orderBy("year")

display(df_result)

#Task 5: Persist and query 
1.Write the Task 4 result to a Delta table in Unity Catalog, in the catalog and schema your interviewer gives you. Call the table very_old_90_over_split.


In [0]:
%python
write_delta_table(df_result, very_old_90_over_split)

I save the gold result as a Delta table so it’s stored permanently and can be queried with SQL. I store the percentages as numbers rather than display text, so they can be sorted, compared and aggregated correctly

In [0]:
%sql
select * from assesment.gold.very_old_90_over_split

Using SQL, return the year in which males made up the largest share of the 90 and over population.

In [0]:
%sql
-- Year in which males made up the largest share of the 90 and over population
SELECT
    year,
    `90_and_over_pcnt_males` AS male_share_pcnt,
    `90_and_over_num_males`  AS males_90_and_over
FROM assesment.gold.very_old_90_over_split
ORDER BY `90_and_over_pcnt_males` DESC
LIMIT 1;

#Answer 5.4
 So, to run this regularly, I’d set it up as a Databricks Job with three steps: bronze, then silver, then gold. Each step just runs one of my notebooks. They’re linked, so if bronze fails, silver and gold don’t run. That way we never end up with half-processed or wrong data sitting in gold.

For when it runs, I could just schedule it daily, weekly or monthly. But for this data I think a file-arrival trigger makes more sense. ONS doesn’t publish that often, so rather than running the job every day for nothing, it just kicks off when a new file lands in the volume.

I’d run it on serverless, so there’s no cluster to look after and we only pay while it’s running. And I’d pass things like the catalog and file name in as parameters, so the same notebooks work in dev, test and prod without touching the code.

If something goes wrong, I’d add a retry for temporary issues and an email or Slack alert so someone knows straight away. My row-count checks would also stop the job if the file looks different from what we expect, so it fails loudly rather than quietly saving wrong numbers. And because everything writes with overwrite, it’s safe to just re-run it.

Same Result With Subquery

In [0]:
%sql
   SELECT year, `90_and_over_pcnt_males` AS male_share_pcnt
   FROM assesment.gold.very_old_90_over_split
   WHERE `90_and_over_pcnt_males` = (SELECT MAX(`90_and_over_pcnt_males`)
                                     FROM assesment.gold.very_old_90_over_split);

#Task 6: Partitioning and distribution

How many partitions is the DataFrame you read in Task 2 spread across? Find out without using the RDD API, and explain why it has that number.

In [0]:
%python
from pyspark.sql import functions as F

# Re-read the raw file exactly as in Task 2
df_raw = read_raw_csv(SOURCE_PATH)

# Count partitions without the RDD API: tag each row with the partition it sits in
num_partitions = df_raw.select(F.spark_partition_id().alias("partition_id")).distinct().count()
print(f"Task 2 DataFrame is spread across {num_partitions} partition(s)")

In [0]:
%python
from pyspark.sql import functions as F

# Step 1: combine the three silver tables into one DataFrame
df_all = (
    spark.table(SILVER_PERSONS_TABLE)
    .unionByName(spark.table(SILVER_MALES_TABLE))
    .unionByName(spark.table(SILVER_FEMALES_TABLE))
)
check_row_count(df_all, 3 * YEARS_PER_POPULATION, "Combined")   # 63 rows

# Step 2: repartition into 3 partitions, using population as the key
df_repartitioned = df_all.repartition(3, "population")

# Step 3: show how many rows of each population landed in each partition
display(
    df_repartitioned
    .groupBy(F.spark_partition_id().alias("partition_id"), "population")
    .count()
    .orderBy("partition_id", "population")
)

In [0]:
%python
very_old_by_population = f"{CATALOG}.gold.very_old_by_population"

(
    df_all.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("population")
    .saveAsTable(very_old_by_population)
)
print(f"Saved {spark.table(very_old_by_population).count()} rows to {very_old_by_population}")